# Part 12 · Agent Substrate: build an agent that sleeps between turns

An agent is idle almost all of the time, and a pod charges you for every second of it. Agent Substrate runs each conversation as an **actor**: a gVisor-sandboxed process while it serves a turn, and a snapshot on disk the rest of the time. A small pool of pre-warmed workers does all the running.

In this lab you build that from nothing on the `kind-substrate` cluster: a **WorkerPool** of your own, a new agent on it, the **ActorTemplate** kagent renders for it, and an **AgentHarness** on the same pool. Then you watch one conversation get snapshotted and come back, and put load through the pool until it runs out of workers.

The **App** tab shows the pool live: the workers, every actor with its state and snapshot version, whether a gVisor process is alive for it, and each change as it happens. Keep it open.

**What we will do**

1. **Define a worker pool**: two pre-warmed gVisor workers.
2. **Create an agent** on that pool.
3. **Read the ActorTemplate** kagent rendered from it.
4. **Add a harness** to the same pool.
5. **Watch one turn**, and catch gVisor serving it.
6. **Snapshot a conversation, then bring it back**.
7. **Load it**: concurrency is the worker count.

## 1. Define a worker pool

A **WorkerPool** is how many sandboxes to keep warm, which worker image, and which sandbox class. Each worker runs **one** actor at a time. The pool size is the number of turns that can run at once, not the number of agents that can live on it.

The label matters: the ActorTemplate kagent renders selects workers by `kagent.dev/worker-pool`, not by the pool's name.

In [ ]:
show $SUBSTRATE_YAML/lab-pool.yaml
kubectl --context $CTX apply -f $SUBSTRATE_YAML/lab-pool.yaml
wait-pool lab-pool 2

## 2. Create an agent on the pool

A **SandboxAgent** is the same declarative spec as any kagent Agent (model, system message, tools), plus `substrate.workerPoolRef`. kagent does not start a pod for it. It renders an ActorTemplate, and Substrate boots one golden actor, waits for it to be ready and snapshots it. Every conversation afterwards starts from that snapshot instead of booting the image.

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 900 410" role="img" aria-label="From a SandboxAgent to a running actor: kagent renders an ActorTemplate, substrate builds a golden actor, routes turns to workers, and stores snapshots" font-family="-apple-system,Segoe UI,Roboto,sans-serif">
<rect width="900" height="410" fill="#ffffff"/>
<defs><marker id="a" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker>
<marker id="ag" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#15803d"/></marker>
<marker id="as" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#64748b"/></marker></defs>
<text x="450" y="28" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">From a SandboxAgent to a running actor</text>
<rect x="20" y="52" width="860" height="112" rx="10" fill="#f5f3ff" stroke="#7c3aed" stroke-width="1.5"/>
<text x="34" y="72" font-size="11" font-weight="700" fill="#5b21b6">kagent (control plane)</text>
<rect x="40" y="86" width="150" height="58" rx="8" fill="#ffffff" stroke="#7c3aed" stroke-width="1.5"/>
<text x="115" y="110" text-anchor="middle" font-size="12" font-weight="700" fill="#312e81">SandboxAgent</text>
<text x="115" y="128" text-anchor="middle" font-size="10" fill="#4338ca">kagent.dev/v1alpha2</text>
<line x1="190" y1="115" x2="238" y2="115" stroke="#334155" stroke-width="1.6" marker-end="url(#a)"/>
<rect x="240" y="86" width="170" height="58" rx="8" fill="#ffffff" stroke="#7c3aed" stroke-width="1.5"/>
<text x="325" y="110" text-anchor="middle" font-size="12" font-weight="700" fill="#312e81">kagent-controller</text>
<text x="325" y="128" text-anchor="middle" font-size="10" fill="#4338ca">renders the ActorTemplate</text>
<line x1="410" y1="115" x2="458" y2="115" stroke="#334155" stroke-width="1.6" marker-end="url(#a)"/>
<rect x="460" y="86" width="180" height="58" rx="8" fill="#ffffff" stroke="#7c3aed" stroke-width="1.5"/>
<text x="550" y="110" text-anchor="middle" font-size="12" font-weight="700" fill="#312e81">ActorTemplate</text>
<text x="550" y="128" text-anchor="middle" font-size="10" fill="#4338ca">image, env, snapshot location</text>
<rect x="690" y="86" width="170" height="58" rx="8" fill="#ffffff" stroke="#7c3aed" stroke-width="1.5"/>
<text x="775" y="110" text-anchor="middle" font-size="12" font-weight="700" fill="#312e81">A2A request</text>
<text x="775" y="128" text-anchor="middle" font-size="10" fill="#4338ca">/api/a2a-sandboxes/ns/name/</text>
<rect x="20" y="184" width="860" height="82" rx="10" fill="#ecfeff" stroke="#0891b2" stroke-width="1.5"/>
<text x="34" y="204" font-size="11" font-weight="700" fill="#155e75">Agent Substrate (API group ate.dev)</text>
<rect x="40" y="214" width="200" height="42" rx="8" fill="#ffffff" stroke="#0891b2" stroke-width="1.5"/>
<text x="140" y="240" text-anchor="middle" font-size="12" font-weight="700" fill="#164e63">ate-controller</text>
<rect x="260" y="214" width="200" height="42" rx="8" fill="#ffffff" stroke="#0891b2" stroke-width="1.5"/>
<text x="360" y="240" text-anchor="middle" font-size="12" font-weight="700" fill="#164e63">object storage</text>
<rect x="480" y="214" width="180" height="42" rx="8" fill="#ffffff" stroke="#0891b2" stroke-width="1.5"/>
<text x="570" y="240" text-anchor="middle" font-size="12" font-weight="700" fill="#164e63">ate-api-server</text>
<rect x="680" y="214" width="180" height="42" rx="8" fill="#ffffff" stroke="#0891b2" stroke-width="1.5"/>
<text x="770" y="240" text-anchor="middle" font-size="12" font-weight="700" fill="#164e63">atenet-router</text>
<line x1="550" y1="144" x2="550" y2="212" stroke="#334155" stroke-width="1.6" marker-end="url(#a)"/>
<text x="558" y="182" font-size="10" fill="#475569">golden actor</text>
<line x1="775" y1="144" x2="775" y2="212" stroke="#334155" stroke-width="1.6" marker-end="url(#a)"/>
<text x="783" y="182" font-size="10" fill="#475569">resume, route</text>
<rect x="20" y="286" width="860" height="112" rx="10" fill="#f0fdf4" stroke="#15803d" stroke-width="1.5"/>
<text x="34" y="306" font-size="11" font-weight="700" fill="#14532d">WorkerPool lab-pool</text>
<rect x="40" y="326" width="260" height="56" rx="8" fill="#ffffff" stroke="#64748b" stroke-width="1.5" stroke-dasharray="5 3"/>
<text x="170" y="348" text-anchor="middle" font-size="12" font-weight="700" fill="#334155">every idle actor</text>
<text x="170" y="368" text-anchor="middle" font-size="10" fill="#475569">suspended: a snapshot, no process</text>
<rect x="320" y="326" width="260" height="56" rx="8" fill="#ffffff" stroke="#15803d" stroke-width="1.5"/>
<text x="450" y="348" text-anchor="middle" font-size="12" font-weight="700" fill="#14532d">worker pod 1 · ateom-gvisor</text>
<text x="450" y="368" text-anchor="middle" font-size="10" fill="#166534">one session's actor, running in runsc</text>
<rect x="600" y="326" width="260" height="56" rx="8" fill="#ffffff" stroke="#15803d" stroke-width="1.5"/>
<text x="730" y="348" text-anchor="middle" font-size="12" font-weight="700" fill="#14532d">worker pod 2 · ateom-gvisor</text>
<text x="730" y="368" text-anchor="middle" font-size="10" fill="#166534">another session's actor</text>
<line x1="280" y1="256" x2="280" y2="324" stroke="#64748b" stroke-width="1.6" marker-end="url(#as)"/>
<text x="288" y="294" font-size="10" fill="#475569">snapshots</text>
<line x1="550" y1="256" x2="550" y2="324" stroke="#15803d" stroke-width="1.6" marker-end="url(#ag)"/>
<text x="558" y="294" font-size="10" fill="#166534">place, resume</text>
<line x1="775" y1="256" x2="775" y2="324" stroke="#15803d" stroke-width="1.6" marker-end="url(#ag)"/>
<text x="783" y="294" font-size="10" fill="#166534">the turn</text>
</svg>

In [ ]:
show $SUBSTRATE_YAML/lab-agent.yaml
time-ready sandboxagent lab-agent $SUBSTRATE_YAML/lab-agent.yaml

## 3. The ActorTemplate kagent rendered

The **ActorTemplate** (`actortemplates.ate.dev`) is what Substrate actually runs, and here it is an ordinary Kubernetes object you can read. kagent created it from the SandboxAgent. Its name is the agent's name plus a hash of its shape, so changing where or how the agent runs produces a second template beside the first rather than editing it.

What to look for in the spec:

- `sandboxClass: gvisor`: every actor from this template runs in its own gVisor sandbox.
- `workerSelector`: only workers labelled `kagent.dev/worker-pool: lab-pool` may run it.
- `image`: the Go ADK agent runtime, pinned by digest.
- `env`: the agent's config (`KAGENT_CONFIG_JSON`: model, prompt, tools) and its model key, all read from Secrets.
- `readyz`: how Substrate knows the golden actor is ready to snapshot.
- the `/data` mount: the directory each snapshot carries from one turn to the next.

And in the status, the **golden snapshot**: the first snapshot, taken once the golden actor passed its readiness check. Every conversation starts from it.

In [ ]:
kubectl --context $CTX -n kagent get actortemplate -l kagent.dev/sandbox-agent=lab-agent

In [ ]:
kubectl --context $CTX -n kagent get actortemplate -l kagent.dev/sandbox-agent=lab-agent -o jsonpath='{.items[0].spec}' \
  | jq '{sandboxClass, workerSelector, image: .containers[0].image, env: [.containers[0].env[].name],
         readyz: .containers[0].readyz, snapshotted: .containers[0].volumeMounts}'

In [ ]:
kubectl --context $CTX -n kagent get actortemplate -l kagent.dev/sandbox-agent=lab-agent -o jsonpath='{.items[0].status}' | jq

## 4. Add a harness to the same pool

A SandboxAgent is an agent kagent defines. An **AgentHarness** is a whole agent runtime, here OpenClaw, that kagent runs on Substrate the same way: its own ActorTemplate, its own golden snapshot, on the pool you name. One shared actor serves all of its chats.

Two templates on one pool is the point: the pool is shared capacity, and anything kagent can render runs on it.

In [ ]:
show $SUBSTRATE_YAML/lab-harness.yaml
kubectl --context $CTX apply -f $SUBSTRATE_YAML/lab-harness.yaml
wait-harness lab-harness
kubectl --context $CTX -n kagent get actortemplate \
  -o custom-columns='TEMPLATE:.metadata.name,OWNER:.metadata.ownerReferences[0].kind,POOL:.spec.workerSelector.matchLabels.kagent\.dev/worker-pool,PHASE:.status.phase' \
  | grep -E '^TEMPLATE|lab-pool'   # every template that runs on lab-pool

## 5. Watch one turn

Every conversation gets its own actor. A turn resumes it onto a free worker, runs it, and suspends it again as soon as the reply is back, and the snapshot version goes up. There is no idle timer: the worker is free the moment the answer is.

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 900 220" role="img" aria-label="Actor states around one turn: Suspended, Resuming, Running, Suspending, then Suspended with the snapshot version incremented" font-family="-apple-system,Segoe UI,Roboto,sans-serif">
<rect width="900" height="220" fill="#ffffff"/>
<defs><marker id="b" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L7,3 L0,6 Z" fill="#334155"/></marker></defs>
<text x="450" y="28" text-anchor="middle" font-size="15" font-weight="700" fill="#0f172a">One turn, as kagent's status API reports it</text>
<rect x="20" y="70" width="150" height="60" rx="8" fill="#e2e8f0" stroke="#64748b" stroke-width="1.5"/>
<text x="95" y="96" text-anchor="middle" font-size="13" font-weight="700" fill="#334155">Suspended</text>
<text x="95" y="116" text-anchor="middle" font-size="10" fill="#475569">snapshot in storage</text>
<line x1="170" y1="100" x2="216" y2="100" stroke="#334155" stroke-width="1.6" marker-end="url(#b)"/>
<rect x="218" y="70" width="150" height="60" rx="8" fill="#fef3c7" stroke="#d97706" stroke-width="1.5"/>
<text x="293" y="96" text-anchor="middle" font-size="13" font-weight="700" fill="#92400e">Resuming</text>
<text x="293" y="116" text-anchor="middle" font-size="10" fill="#b45309">worker chosen, restore</text>
<line x1="368" y1="100" x2="414" y2="100" stroke="#334155" stroke-width="1.6" marker-end="url(#b)"/>
<rect x="416" y="70" width="150" height="60" rx="8" fill="#dcfce7" stroke="#15803d" stroke-width="1.5"/>
<text x="491" y="96" text-anchor="middle" font-size="13" font-weight="700" fill="#14532d">Running</text>
<text x="491" y="116" text-anchor="middle" font-size="10" fill="#166534">runsc alive, turn served</text>
<line x1="566" y1="100" x2="612" y2="100" stroke="#334155" stroke-width="1.6" marker-end="url(#b)"/>
<rect x="614" y="70" width="150" height="60" rx="8" fill="#fef3c7" stroke="#d97706" stroke-width="1.5"/>
<text x="689" y="96" text-anchor="middle" font-size="13" font-weight="700" fill="#92400e">Suspending</text>
<text x="689" y="116" text-anchor="middle" font-size="10" fill="#b45309">checkpoint, upload</text>
<line x1="764" y1="100" x2="810" y2="100" stroke="#334155" stroke-width="1.6" marker-end="url(#b)"/>
<rect x="812" y="70" width="70" height="60" rx="8" fill="#e2e8f0" stroke="#64748b" stroke-width="1.5"/>
<text x="847" y="96" text-anchor="middle" font-size="11" font-weight="700" fill="#334155">version</text>
<text x="847" y="116" text-anchor="middle" font-size="11" font-weight="700" fill="#334155">+1</text>
<text x="450" y="170" text-anchor="middle" font-size="11" fill="#475569">Measured on kind: Resuming for about a second, Running for the length of the model call, Suspended again within eight seconds of the reply.</text>
<text x="450" y="192" text-anchor="middle" font-size="11" fill="#475569">The next turn in the same session resumes from the actor's own snapshot, so the conversation state travels with it.</text>
</svg>

In [ ]:
watch-turn lab-agent "In two sentences, what does it mean to snapshot a running process?"

### Run it in a loop, and watch it on the App tab

One conversation, turn after turn. Each turn wakes the same actor from its last snapshot, runs it in a real gVisor sandbox (the step catches `runsc-sandbox` on the node while the turn is in flight) and suspends it again, one snapshot version higher. Between turns there is a short pause with nothing running at all.

Set **Turns** beside the Run button, open the **App** tab, then run it: the worker lights up for each turn, the actor goes Resuming, Running, Suspended, and its version climbs.

In [ ]:
turn-loop ${TURNS:-5} lab-agent      # TURNS comes from the field beside Run

## 6. Snapshot a conversation, then bring it back

One conversation, three steps apart. Tell the agent something, look at what is left on the node once it has suspended, then ask for it back. Between the turns there is no process at all, only the actor's snapshot. The second turn resumes from that snapshot, so the conversation carries on where it stopped.

In [ ]:
lab-session lab-agent
lab-ask "Remember this number for later: 4242. Just confirm you have it."

In [ ]:
lab-actor
lab-snapshots

In [ ]:
lab-ask "What number did I ask you to remember?"

## 7. Load it: concurrency is the worker count

A worker runs one actor at a time, so the pool size is how many turns can run at once. First the limit: two workers, three turns at the same moment. Two are answered and the third is turned away straight away, because this build refuses rather than queues.

Then scale the pool and keep it busy. Each load client is its own conversation, so its own actor, sending turns back to back. When every worker is busy a turn is refused for a moment and retried, so the workers never sit idle. Watch the App tab: the bays light up, actors are handed a worker, run, suspend and give it back, over and over.

In [ ]:
fire 3 lab-agent             # three turns at once, two workers

In [ ]:
grow-pool ${WORKERS:-6}      # kubectl scale workerpool/lab-pool, then wait until Substrate can use them

In [ ]:
load-run ${CONVERSATIONS:-8} ${DURATION:-45} lab-agent

## Reset

**Reset lab** deletes the agent, the harness and the pool. The default pool, the other agents on the cluster and Substrate itself stay up.